In [1]:
pip install bs4 lxml unidecode MLB-StatsAPI --quiet

Note: you may need to restart the kernel to use updated packages.


DEPRECATION: dropbox 11.36.0 has a non-standard dependency specifier stone>=2.*. pip 23.3 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of dropbox or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 23.2.1 -> 23.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
from bs4 import BeautifulSoup, Comment
import requests
import pandas as pd
import numpy as np
import re
from lxml import html
import datetime
import pdb
import time
import os
from unidecode import unidecode
import statsapi

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Edit these before running to control which seasons are collected.

YEARS = [2021, 2022, 2023]   # seasons to collect — add 2024, 2025 as needed

# Season start/end dates per year (MM-DD format)
SEASON_DATES = {
    2021: ('05-01', '10-03'),
    2022: ('04-07', '10-05'),
    2023: ('03-30', '10-01'),
    2024: ('03-20', '09-29'),
    2025: ('03-27', '09-28'),
}

# All-Star break dates to skip (statsapi returns no games these days)
ALL_STAR_BREAKS = {
    2021: ['2021-07-12', '2021-07-13', '2021-07-14', '2021-07-15'],
    2022: ['2022-07-18', '2022-07-19', '2022-07-20', '2022-07-21'],
    2023: ['2023-07-10', '2023-07-11', '2023-07-12', '2023-07-13'],
    2024: ['2024-07-15', '2024-07-16', '2024-07-17', '2024-07-18'],
    2025: ['2025-07-14', '2025-07-15', '2025-07-16', '2025-07-17'],
}

# Output file — Google Drive path for Colab, or local path
OUTPUT_FILE = '/content/drive/MyDrive/NRFI_collected.csv'

# Note: OAK relocated to Sacramento in 2025.
# When collecting 2025 data, change team_abbvs['OAK']['team_pct'] to 'Sacramento'.

# ── Configuration ─────────────────────────────────────────────────────────────
# Edit these before running to control which seasons are collected.

YEARS = [2021, 2022, 2023]   # seasons to collect — add 2024, 2025 as needed

# Season start/end dates per year (MM-DD format)
SEASON_DATES = {
    2021: ('05-01', '10-03'),
    2022: ('04-07', '10-05'),
    2023: ('03-30', '10-01'),
    2024: ('03-20', '09-29'),
    2025: ('03-27', '09-28'),
}

# All-Star break dates to skip (statsapi returns no games these days)
ALL_STAR_BREAKS = {
    2021: ['2021-07-12', '2021-07-13', '2021-07-14', '2021-07-15'],
    2022: ['2022-07-18', '2022-07-19', '2022-07-20', '2022-07-21'],
    2023: ['2023-07-10', '2023-07-11', '2023-07-12', '2023-07-13'],
    2024: ['2024-07-15', '2024-07-16', '2024-07-17', '2024-07-18'],
    2025: ['2025-07-14', '2025-07-15', '2025-07-16', '2025-07-17'],
}

# Output file — Google Drive path for Colab, or local path
OUTPUT_FILE = '/content/drive/MyDrive/NRFI_collected.csv'

# Note: OAK relocated to Sacramento in 2025 — handled automatically in the loop.

In [ ]:
def digit(n):
    if(n < 10):
        return '0' + str(n)
    else:
        return str(n)

def lineup_to_ops(lineup):
    lineup_ops = []
    for batter in lineup:
        try:
            batter_ops = ops_df[ops_df['PlayerName'] == unidecode(batter)]['OPS'].item()
            lineup_ops.append(batter_ops)
        except:
            continue
    try:
        ops = round(sum(lineup_ops) / len(lineup_ops), 3)
    except:
        return None
    return ops

def get_whip(pitcher_name):
    whip = whip_df[whip_df['PlayerName'] == pitcher_name]['WHIP']
    if(len(whip) == 1):
        return whip.item()
    else:
        return None

def api_name(personId):
    return statsapi.get('people', {'personIds':personId})['people'][0]['fullName']

def get_lineup(boxscore, home_away):
    hitters, i = 0, 1
    lineup = []
    while hitters < 4:
        id = boxscore[home_away+'Batters'][i]['personId']
        sub = boxscore[home_away+'Batters'][i]['substitution']
        if(not sub):
            hitters += 1
            name = api_name(id)
            lineup.append(name)
        i += 1
    return lineup

team_abbvs = {
    'PHI':{'team_pct':'Philadelphia', 'normal':'PHI'},
    'SF':{'team_pct':'SF Giants', 'normal':'SF'},
    'TEX':{'team_pct':'Texas', 'normal':'TEX'},
    'BOS':{'team_pct':'Boston', 'normal':'BOS'},
    'KC':{'team_pct':'Kansas City', 'normal':'KC'},
    'DET':{'team_pct':'Detroit', 'normal':'DET'},
    'NYY':{'team_pct':'NY Yankees', 'normal':'NYY'},
    'TB':{'team_pct':'Tampa Bay', 'normal':'TB'},
    'TOR':{'team_pct':'Toronto', 'normal':'TOR'},
    'PIT':{'team_pct':'Pittsburgh', 'normal':'PIT'},
    'OAK':{'team_pct':'Oakland', 'normal':'OAK'},
    'BAL':{'team_pct':'Baltimore', 'normal':'BAL'},
    'WSH':{'team_pct':'Washington', 'normal':'WAS'},
    'NYM':{'team_pct':'NY Mets', 'normal':'NYM'},
    'MIN':{'team_pct':'Minnesota', 'normal':'MIN'},
    'CWS':{'team_pct':'Chi Sox', 'normal':'CHW'},
    'SEA':{'team_pct':'Seattle', 'normal':'SEA'},
    'CLE':{'team_pct':'Cleveland', 'normal':'CLE'},
    'CHC':{'team_pct':'Chi Cubs', 'normal':'CHC'},
    'STL':{'team_pct':'St. Louis', 'normal':'STL'},
    'MIA':{'team_pct':'Miami', 'normal':'MIA'},
    'ATL':{'team_pct':'Atlanta', 'normal':'ATL'},
    'MIL':{'team_pct':'Milwaukee', 'normal':'MIL'},
    'AZ':{'team_pct':'Arizona', 'normal':'ARI'},
    'HOU':{'team_pct':'Houston', 'normal':'HOU'},
    'LAA':{'team_pct':'LA Angels', 'normal':'LAA'},
    'SD':{'team_pct':'San Diego', 'normal':'SD'},
    'LAD':{'team_pct':'LA Dodgers', 'normal':'LAD'},
    'CIN':{'team_pct':'Cincinnati', 'normal':'CIN'},
    'COL':{'team_pct':'Colorado', 'normal':'COL'}
}

park_factor_df = pd.DataFrame(
    {'Team': {0: 'COL', 1: 'CIN', 2: 'BOS', 3: 'LAA', 4: 'PHI', 5: 'KC', 6: 'CHW', 7: 'LAD', 8: 'BAL', 9: 'ARI', 10: 'PIT', 11: 'MIL', 12: 'SF', 13: 'ATL', 14: 'WAS', 15: 'CLE', 16: 'TOR', 17: 'MIA', 18: 'TEX', 19: 'NYY', 20: 'CHC', 21: 'HOU', 22: 'MIN', 23: 'DET', 24: 'TB', 25: 'NYM', 26: 'STL', 27: 'OAK', 28: 'SD', 29: 'SEA'},
     'Park Factor': {0: 112, 1: 111, 2: 109, 3: 104, 4: 104, 5: 103, 6: 102, 7: 102, 8: 101, 9: 101, 10: 100, 11: 100, 12: 100, 13: 100, 14: 100, 15: 99, 16: 99, 17: 99, 18: 99, 19: 99, 20: 98, 21: 98, 22: 98, 23: 97, 24: 96, 25: 96, 26: 95, 27: 94, 28: 94, 29: 91}}
)
park_factor_dict = park_factor_df.set_index('Team').to_dict()['Park Factor']

day_night_to_int = {'Day':0, 'Night':1}

# ── Collection Loop ────────────────────────────────────────────────────────────
# Uses YEARS, SEASON_DATES, ALL_STAR_BREAKS, OUTPUT_FILE from the config cell.
# Home/Away YRFI splits are fetched directly (home_yrfi_pct uses 'Home' column,
# away_yrfi_pct uses 'Away' column) — no post-processing patch needed.

big_df = pd.DataFrame({})

for year in YEARS:
    start_str, end_str = SEASON_DATES[year]
    day1      = datetime.date(year, int(start_str[:2]), int(start_str[3:]))
    day_final = datetime.date(year, int(end_str[:2]),   int(end_str[3:]))
    num_days  = (day_final - day1).days
    year_breaks = ALL_STAR_BREAKS.get(year, [])

    # Team name overrides for relocations
    team_abbvs['OAK']['team_pct'] = 'Sacramento' if year >= 2025 else 'Oakland'

    print(f'\n=== Collecting {year}: {day1} to {day_final} ({num_days} days) ===')

    for delta in range(num_days):
        date = day1 + datetime.timedelta(delta)
        yesterday = date - datetime.timedelta(1)

        if str(date) in year_breaks:
            print('All Star Break')
            continue

        date_minus_thirty_days = date - datetime.timedelta(30)
        date_minus_sixty_days  = date - datetime.timedelta(60)
        season_start_date      = datetime.date(year, 3, 1)

        team_pct = pd.read_html('https://www.teamrankings.com/mlb/stat/yes-run-first-inning-pct?date='+str(yesterday))[0]

        ops_url = 'https://www.fangraphs.com/api/leaders/major-league/data?age=&pos=all&stats=bat&lg=all&qual=1&season='+str(date.year)+'&season1='+str(date.year)+'&startdate='+str(date_minus_thirty_days)+'&enddate='+str(yesterday)+'&month=1000&hand=&team=0&pageitems=2000000000&pagenum=1&ind=0&rost=0&players=&type=8&postseason=&sortdir=default&sortstat=WAR'
        r = requests.get(ops_url)
        ops_df = pd.json_normalize(r.json()['data'])[['PlayerName', 'OPS']]
        ops_df['cleanedName'] = ops_df['PlayerName'].apply(unidecode)

        payload = {"strPlayerId":"all","strSplitArr":[44],"strGroup":"season","strPosition":"P","strType":"1","strStartDate":str(season_start_date),"strEndDate":str(yesterday),"strSplitTeams":False,"dctFilters":[{"stat": "IP","comp": "gt","low": "2","high": -99,"label": "IP >= 2.0","value": 0}],"strStatType":"player","strAutoPt":"true","arrPlayerId":[],"strSplitArrPitch":[],"arrWxTemperature":None,"arrWxPressure":None,"arrWxAirDensity":None,"arrWxElevation":None,"arrWxWindSpeed":None}
        pitcher_runs_url = 'https://www.fangraphs.com/api/leaders/splits/splits-leaders'
        r = requests.post(pitcher_runs_url, json=payload)
        try:
            pitcher_runs_df = pd.json_normalize(r.json()['data'])
            pitcher_runs_df['RA'] = pitcher_runs_df['R'] / pitcher_runs_df['G']
            pitcher_runs_df['cleanedName'] = pitcher_runs_df['playerName'].apply(unidecode)
        except:
            print('couldn\'t load pitcher runs df')

        whip_url = 'https://www.fangraphs.com/api/leaders/major-league/data?age=0&pos=all&stats=pit&lg=all&qual=1&season='+str(date.year)+'&season1='+str(date.year)+'&startdate='+str(date_minus_sixty_days)+'&enddate='+str(yesterday)+'&month=1000&hand=&team=0&pageitems=100000&pagenum=1&ind=0&rost=0&players=0&type=1&postseason=&sortdir=default&sortstat=SIERA&qual=1'
        r = requests.get(whip_url)
        whip_df = pd.json_normalize(r.json()['data'])
        whip_df['cleanedName'] = whip_df['PlayerName'].apply(unidecode)

        games = statsapi.schedule(start_date=date.strftime('%m/%d/%Y'))
        day = []
        ids = []
        if(len(games) == 0):
            continue
        for g in games:
            game = {}
            game_id = g['game_id']
            away_abbreviation = statsapi.get('game', params={'gamePk':game_id})['gameData']['teams']['away']['abbreviation']
            home_abbreviation = statsapi.get('game', params={'gamePk':game_id})['gameData']['teams']['home']['abbreviation']

            id = str(date)+'-'+team_abbvs[away_abbreviation]['normal']+'@'+team_abbvs[home_abbreviation]['normal']
            if(id) in ids:
                id += '-game2'
            game['id'] = id
            ids.append(id)

            linescore = statsapi.linescore(game_id)
            top1 = int(re.search('[0-9]+', linescore.split('\n')[1]).group())
            bot1 = int(re.search('[0-9]+', linescore.split('\n')[2]).group())
            first_inn_runs = int(top1) + int(bot1)
            game['1st_runs'] = first_inn_runs

            boxscore = statsapi.boxscore_data(game_id)

            away_lineup = get_lineup(boxscore, 'away')
            home_lineup = get_lineup(boxscore, 'home')
            away_ops = lineup_to_ops(away_lineup)
            home_ops = lineup_to_ops(home_lineup)
            game['away_lineup'] = away_lineup
            game['home_lineup'] = home_lineup
            game['away_ops'] = away_ops
            game['home_ops'] = home_ops

            away_pitcher_id = boxscore['awayPitchers'][1]['personId']
            away_pitcher = api_name(away_pitcher_id)
            home_pitcher_id = boxscore['homePitchers'][1]['personId']
            home_pitcher = api_name(home_pitcher_id)

            try:
                away_pitcher_first_runs = pitcher_runs_df[pitcher_runs_df['cleanedName'] == away_pitcher]['RA'].item()
            except:
                away_pitcher_first_runs = None
            try:
                home_pitcher_first_runs = pitcher_runs_df[pitcher_runs_df['cleanedName'] == home_pitcher]['RA'].item()
            except:
                home_pitcher_first_runs = None

            game['away_pitcher'] = away_pitcher
            game['home_pitcher'] = home_pitcher
            game['away_pitcher_ra'] = away_pitcher_first_runs
            game['home_pitcher_ra'] = home_pitcher_first_runs

            game['away_whip'] = get_whip(away_pitcher)
            game['home_whip'] = get_whip(home_pitcher)

            # YRFI pct — Home/Away splits from teamrankings (not overall year rate)
            home_yrfi_pct = round(float(team_pct[team_pct['Team'] == team_abbvs[home_abbreviation]['team_pct']]['Home'].item()[:-1])/100, 4)
            game['home_yrfi_pct'] = home_yrfi_pct
            away_yrfi_pct = round(float(team_pct[team_pct['Team'] == team_abbvs[away_abbreviation]['team_pct']]['Away'].item()[:-1])/100, 4)
            game['away_yrfi_pct'] = away_yrfi_pct

            park_factor = park_factor_dict[team_abbvs[home_abbreviation]['normal']]
            game['park_factor'] = park_factor

            game_box_info = boxscore['gameBoxInfo']
            for line in game_box_info:
                if(line['label'] == 'Weather'):
                    game['weather'] = line['value']
                if(line['label'] == 'First pitch'):
                    game['time'] = line['value']

            day.append(game)
            print(game['id'])

        big_df = pd.concat([big_df, pd.DataFrame(day)])
        big_df = big_df.reset_index(drop=True)
        big_df.to_csv(OUTPUT_FILE)  # save incrementally each day

print(f'\nCollection complete. {len(big_df)} games saved to {OUTPUT_FILE}')

# Single Day

In [12]:
def digit(n):
    if(n < 10):
        return '0' + str(n)
    else:
        return str(n)

def lineup_to_ops(lineup, ops_df):
    lineup_ops = []
    for batter in lineup:
        try:
            batter_ops = ops_df[ops_df['PlayerName'] == unidecode(batter)]['OPS'].item()
            lineup_ops.append(batter_ops)
        except:
            continue
    try:
        ops = round(sum(lineup_ops) / len(lineup_ops), 3)
    except:
        return None
    return ops

def get_whip(pitcher_name, whip_df):
    whip = whip_df[whip_df['PlayerName'] == pitcher_name]['WHIP']
    if(len(whip) == 1):
        return whip.item()
    else:
        return None

def api_name(personId):
    return statsapi.get('people', {'personIds':personId})['people'][0]['fullName']

def get_lineup(boxscore, home_away):
    hitters, i = 0, 1
    lineup = []
    while hitters < 4:
        id = boxscore[home_away+'Batters'][i]['personId']
        sub = boxscore[home_away+'Batters'][i]['substitution']
        if(not sub):
            hitters += 1
            name = api_name(id)
            lineup.append(name)
        i += 1
    return lineup

def main():
    team_abbvs = {
        'PHI':{'team_pct':'Philadelphia', 'normal':'PHI'},
        'SF':{'team_pct':'SF Giants', 'normal':'SF'},
        'TEX':{'team_pct':'Texas', 'normal':'TEX'},
        'BOS':{'team_pct':'Boston', 'normal':'BOS'},
        'KC':{'team_pct':'Kansas City', 'normal':'KC'},
        'DET':{'team_pct':'Detroit', 'normal':'DET'},
        'NYY':{'team_pct':'NY Yankees', 'normal':'NYY'},
        'TB':{'team_pct':'Tampa Bay', 'normal':'TB'},
        'TOR':{'team_pct':'Toronto', 'normal':'TOR'},
        'PIT':{'team_pct':'Pittsburgh', 'normal':'PIT'},
        'OAK':{'team_pct':'Oakland', 'normal':'OAK'},
        'BAL':{'team_pct':'Baltimore', 'normal':'BAL'},
        'WSH':{'team_pct':'Washington', 'normal':'WAS'},
        'NYM':{'team_pct':'NY Mets', 'normal':'NYM'},
        'MIN':{'team_pct':'Minnesota', 'normal':'MIN'},
        'CWS':{'team_pct':'Chi Sox', 'normal':'CHW'},
        'SEA':{'team_pct':'Seattle', 'normal':'SEA'},
        'CLE':{'team_pct':'Cleveland', 'normal':'CLE'},
        'CHC':{'team_pct':'Chi Cubs', 'normal':'CHC'},
        'STL':{'team_pct':'St. Louis', 'normal':'STL'},
        'MIA':{'team_pct':'Miami', 'normal':'MIA'},
        'ATL':{'team_pct':'Atlanta', 'normal':'ATL'},
        'MIL':{'team_pct':'Milwaukee', 'normal':'MIL'},
        'AZ':{'team_pct':'Arizona', 'normal':'ARI'},
        'HOU':{'team_pct':'Houston', 'normal':'HOU'},
        'LAA':{'team_pct':'LA Angels', 'normal':'LAA'},
        'SD':{'team_pct':'San Diego', 'normal':'SD'},
        'LAD':{'team_pct':'LA Dodgers', 'normal':'LAD'},
        'CIN':{'team_pct':'Cincinnati', 'normal':'CIN'},
        'COL':{'team_pct':'Colorado', 'normal':'COL'}
    }

    park_factor_df = pd.DataFrame(
        {'Team': {0: 'COL', 1: 'CIN', 2: 'BOS', 3: 'LAA', 4: 'PHI', 5: 'KC', 6: 'CHW', 7: 'LAD', 8: 'BAL', 9: 'ARI', 10: 'PIT', 11: 'MIL', 12: 'SF', 13: 'ATL', 14: 'WAS', 15: 'CLE', 16: 'TOR', 17: 'MIA', 18: 'TEX', 19: 'NYY', 20: 'CHC', 21: 'HOU', 22: 'MIN', 23: 'DET', 24: 'TB', 25: 'NYM', 26: 'STL', 27: 'OAK', 28: 'SD', 29: 'SEA'},
        'Park Factor': {0: 112, 1: 111, 2: 109, 3: 104, 4: 104, 5: 103, 6: 102, 7: 102, 8: 101, 9: 101, 10: 100, 11: 100, 12: 100, 13: 100, 14: 100, 15: 99, 16: 99, 17: 99, 18: 99, 19: 99, 20: 98, 21: 98, 22: 98, 23: 97, 24: 96, 25: 96, 26: 95, 27: 94, 28: 94, 29: 91}}
    )
    park_factor_dict = park_factor_df.set_index('Team').to_dict()['Park Factor']

    day_night_to_int = {'Day':0, 'Night':1}
    big_df = pd.DataFrame({})

    # ********************** Day Loop **********************************
    m, d, y = 9, 17, 2019 # start date

    # start with May 1 for all seasons
    # last day of seasons: 10-1-23, 10-5-22, 10-3-21

    date = datetime.date(y, m, d)
    yesterday = date - datetime.timedelta(1)
    req = 1

    all_star_breaks = ['2021-07-12', '2021-07-13', '2021-07-14', '2021-07-15',
                        '2022-07-18', '2022-07-19', '2022-07-20', '2022-07-21',
                        '2023-07-10', '2023-07-11', '2023-07-12', '2023-07-13']

    if(str(date) in all_star_breaks):
        return('All Star Break')
    
    date_minus_thirty_days = date - datetime.timedelta(30)
    date_minus_sixty_days = date - datetime.timedelta(60)
    season_start_date = datetime.date(y, 3, 1)

    team_pct = pd.read_html('https://www.teamrankings.com/mlb/stat/yes-run-first-inning-pct?date='+str(yesterday))[0]
    ops_url = 'https://www.fangraphs.com/api/leaders/major-league/data?age=&pos=all&stats=bat&lg=all&qual=1&season='+str(date.year)+'&season1='+str(date.year)+'&startdate='+str(date_minus_thirty_days)+'&enddate='+str(yesterday)+'&month=1000&hand=&team=0&pageitems=2000000000&pagenum=1&ind=0&rost=0&players=&type=8&postseason=&sortdir=default&sortstat=WAR'
    r = requests.get(ops_url)
    ops_df = pd.json_normalize(r.json()['data'])[['PlayerName', 'OPS']]
    ops_df['cleanedName'] = ops_df['PlayerName'].apply(unidecode)
    payload = {"strPlayerId":"all","strSplitArr":[44],"strGroup":"season","strPosition":"P","strType":"1","strStartDate":str(season_start_date),"strEndDate":str(yesterday),"strSplitTeams":False,"dctFilters":[{"stat": "IP","comp": "gt","low": "2","high": -99,"label": "IP â‰¥ 2.0","value": 0}],"strStatType":"player","strAutoPt":"true","arrPlayerId":[],"strSplitArrPitch":[],"arrWxTemperature":None,"arrWxPressure":None,"arrWxAirDensity":None,"arrWxElevation":None,"arrWxWindSpeed":None}
    pitcher_runs_url = 'https://www.fangraphs.com/api/leaders/splits/splits-leaders'
    r = requests.post(pitcher_runs_url, json=payload)
    try:
        pitcher_runs_df = pd.json_normalize(r.json()['data'])
        pitcher_runs_df['RA'] = pitcher_runs_df['R'] / pitcher_runs_df['G']
        pitcher_runs_df['cleanedName'] = pitcher_runs_df['playerName'].apply(unidecode)
    except:
        print('couldn\'t load pitcher runs df')

    whip_url = 'https://www.fangraphs.com/api/leaders/major-league/data?age=0&pos=all&stats=pit&lg=all&qual=1&season='+str(date.year)+'&season1='+str(date.year)+'&startdate='+str(date_minus_sixty_days)+'&enddate='+str(yesterday)+'&month=1000&hand=&team=0&pageitems=100000&pagenum=1&ind=0&rost=0&players=0&type=1&postseason=&sortdir=default&sortstat=SIERA&qual=1'
    r = requests.get(whip_url)
    whip_df = pd.json_normalize(r.json()['data'])
    whip_df['cleanedName'] = whip_df['PlayerName'].apply(unidecode)

    # get list of games as dicts from single day
    games = statsapi.schedule(start_date=date.strftime('%m/%d/%Y'))
    day = []
    ids = []
    if(len(games) == 0): # if no games happen that day
        return 'No games today'
    for g in games: # for each game that day
        game = {}
        game_id = g['game_id']
        # print('loaded game')
        away_abbreviation = statsapi.get('game', params={'gamePk':game_id})['gameData']['teams']['away']['abbreviation']
        home_abbreviation = statsapi.get('game', params={'gamePk':game_id})['gameData']['teams']['home']['abbreviation']

        id = str(date)+'-'+team_abbvs[away_abbreviation]['normal']+'@'+team_abbvs[home_abbreviation]['normal']
        if(id) in ids:
            id += '-game2'
        game['id'] = id
        ids.append(id)

        linescore = statsapi.linescore(game_id)
        top1 = int(re.search('[0-9]+', linescore.split('\n')[1]).group())
        bot1 = int(re.search('[0-9]+', linescore.split('\n')[2]).group())
        first_inn_runs = int(top1) + int(bot1)
        game['1st_runs'] = first_inn_runs

        boxscore = statsapi.boxscore_data(game_id)

        away_lineup = get_lineup(boxscore, 'away')
        home_lineup = get_lineup(boxscore, 'home')
        away_ops = lineup_to_ops(away_lineup, ops_df)
        home_ops = lineup_to_ops(home_lineup, ops_df)
        game['away_lineup'] = away_lineup
        game['home_lineup'] = home_lineup
        game['away_ops'] = away_ops
        game['home_ops'] = home_ops

        away_pitcher_id = boxscore['awayPitchers'][1]['personId']
        away_pitcher = api_name(away_pitcher_id)
        home_pitcher_id = boxscore['homePitchers'][1]['personId']
        home_pitcher = api_name(home_pitcher_id)

        try:
            away_pitcher_first_runs = pitcher_runs_df[pitcher_runs_df['cleanedName'] == away_pitcher]['RA'].item()
        except:
            away_pitcher_first_runs = None
        try:
            home_pitcher_first_runs = pitcher_runs_df[pitcher_runs_df['cleanedName'] == home_pitcher]['RA'].item()
        except:
            home_pitcher_first_runs = None

        game['away_pitcher'] = away_pitcher
        game['home_pitcher'] = home_pitcher
        game['away_pitcher_ra'] = away_pitcher_first_runs
        game['home_pitcher_ra'] = home_pitcher_first_runs

        game['away_whip'] = get_whip(away_pitcher, whip_df)
        game['home_whip'] = get_whip(home_pitcher, whip_df)

        # yrfi pct - year to date
        home_yrfi_pct = round(float(team_pct[team_pct['Team'] == team_abbvs[home_abbreviation]['team_pct']]['Home'].item()[:-1])/100, 4)
        game['home_yrfi_pct'] = home_yrfi_pct
        away_yrfi_pct = round(float(team_pct[team_pct['Team'] == team_abbvs[away_abbreviation]['team_pct']]['Away'].item()[:-1])/100, 4)
        game['away_yrfi_pct'] = away_yrfi_pct

        # Park Factor
        park_factor = park_factor_dict[team_abbvs[home_abbreviation]['normal']]
        game['park_factor'] = park_factor

        game_box_info = boxscore['gameBoxInfo']
        for line in game_box_info:
            if(line['label'] == 'Weather'):
                game['weather'] = line['value']
            if(line['label'] == 'First pitch'):
                game['time'] = line['value']

        day.append(game)

    big_df = pd.concat([big_df, pd.DataFrame(day)])
    big_df = big_df.reset_index(drop=True)
    return big_df

In [13]:
df = main()

In [14]:
df['away_ops']

0     0.81
1     0.92
2     1.03
3     0.86
4     0.73
5     0.82
6     0.90
7     0.68
8     1.00
9     0.72
10    0.62
11    0.94
12    0.76
13    0.82
14    0.95
Name: away_ops, dtype: object

In [ ]:
from decimal import Decimal
float_columns = ['away_ops', 'home_ops', 'away_pitcher_ra', 'home_pitcher_ra', 'away_whip', 'home_whip', 'home_yrfi_pct', 'away_yrfi_pct']
for col in float_columns:
    df[col] = df[col].apply(lambda x: round(x, 4)).apply(Decimal)

In [4]:
import datetime
date = datetime.date(2022, 9, 3)

In [6]:
print(date.year)
print(date.month)
print(date.day)

2022
9
3


In [16]:
df.to_csv('C:/Users/Ryan/Desktop/temp.txt')

In [24]:
datetime.date.today()

datetime.date(2024, 12, 12)

In [25]:
datetime.date

datetime.date

In [21]:
str(date.today())

'2024-12-12'

In [19]:
df.to_csv(index=False)

'id,1st_runs,away_lineup,home_lineup,away_ops,home_ops,away_pitcher,home_pitcher,away_pitcher_ra,home_pitcher_ra,away_whip,home_whip,home_yrfi_pct,away_yrfi_pct,park_factor,weather,time\r\n2019-09-17-LAA@NYY,0,"[\'Brian Goodwin\', \'David Fletcher\', \'Kole Calhoun\', \'Albert Pujols\']","[\'DJ LeMahieu\', \'Aaron Judge\', \'Didi Gregorius\', \'Gleyber Torres\']",0.815,0.871,NoÃ© Ramirez,Luis Severino,,,1.568182115589348,,0.3377,0.3133,99,"74 degrees, Clear.",6:39 PM.\r\n2019-09-17-SEA@PIT,0,"[\'Shed Long Jr.\', \'J.P. Crawford\', \'Kyle Seager\', \'Kyle Lewis\']","[\'Kevin Newman\', \'Adam Frazier\', \'Melky Cabrera\', \'JosÃ© Osuna\']",0.92,0.844,Marco Gonzales,Mitch Keller,0.5806451612903226,1.4444444444444444,1.3695652883760456,1.6923076845485079,0.3333,0.26,100,"76 degrees, Clear.",7:07 PM.\r\n2019-09-17-TOR@BAL,2,"[\'Bo Bichette\', \'Cavan Biggio\', \'Lourdes Gurriel Jr.\', \'Vladimir Guerrero Jr.\']","[\'Jonathan Villar\', \'Dwight Smith Jr.\', \'Trey Mancini\', \'Rio Ruiz\']",1

In [ ]:
print()

id,1st_runs,away_lineup,home_lineup,away_ops,home_ops,away_pitcher,home_pitcher,away_pitcher_ra,home_pitcher_ra,away_whip,home_whip,home_yrfi_pct,away_yrfi_pct,park_factor,weather,time
2019-09-17-LAA@NYY,0,"['Brian Goodwin', 'David Fletcher', 'Kole Calhoun', 'Albert Pujols']","['DJ LeMahieu', 'Aaron Judge', 'Didi Gregorius', 'Gleyber Torres']",0.815,0.871,NoÃ© Ramirez,Luis Severino,,,1.568182115589348,,0.3377,0.3133,99,"74 degrees, Clear.",6:39 PM.
2019-09-17-SEA@PIT,0,"['Shed Long Jr.', 'J.P. Crawford', 'Kyle Seager', 'Kyle Lewis']","['Kevin Newman', 'Adam Frazier', 'Melky Cabrera', 'JosÃ© Osuna']",0.92,0.844,Marco Gonzales,Mitch Keller,0.5806451612903226,1.4444444444444444,1.3695652883760456,1.6923076845485079,0.3333,0.26,100,"76 degrees, Clear.",7:07 PM.
2019-09-17-TOR@BAL,2,"['Bo Bichette', 'Cavan Biggio', 'Lourdes Gurriel Jr.', 'Vladimir Guerrero Jr.']","['Jonathan Villar', 'Dwight Smith Jr.', 'Trey Mancini', 'Rio Ruiz']",1.034,0.776,Ryan Tepera,Chandler Shepherd,,,1.0714287310838

In [18]:
df

,id,1st_runs,away_lineup,home_lineup,away_ops,home_ops,away_pitcher,home_pitcher,away_pitcher_ra,home_pitcher_ra,away_whip,home_whip,home_yrfi_pct,away_yrfi_pct,park_factor,weather,time
0,2019-09-17-LAA@NYY,0,"[Brian Goodwin, David Fletcher, Kole Calhoun, ...","[DJ LeMahieu, Aaron Judge, Didi Gregorius, Gle...",0.810,0.870,NoÃ© Ramirez,Luis Severino,NaN,NaN,1.570,NaN,0.340,0.310,99,"74 degrees, Clear.",6:39 PM.
1,2019-09-17-SEA@PIT,0,"[Shed Long Jr., J.P. Crawford, Kyle Seager, Ky...","[Kevin Newman, Adam Frazier, Melky Cabrera, Jo...",0.920,0.840,Marco Gonzales,Mitch Keller,0.580,1.440,1.370,1.690,0.330,0.260,100,"76 degrees, Clear.",7:07 PM.
2,2019-09-17-TOR@BAL,2,"[Bo Bichette, Cavan Biggio, Lourdes Gurriel Jr...","[Jonathan Villar, Dwight Smith Jr., Trey Manci...",1.030,0.780,Ryan Tepera,Chandler Shepherd,NaN,NaN,1.070,1.570,0.340,0.270,101,"77 degrees, Clear.",7:09 PM.
3,2019-09-17-DET@CLE,1,"[Victor Reyes, Harold Castro, Miguel Cabrera, ...","[Francisco Lindor, Ã“scar Mercado, Carlos Santa...",0.860,0.740,Zac Reininger,Adam Plutko,NaN,0.530,2.060,1.270,0.310,0.260,99,"73 degrees, Clear.",7:10 PM.
4,2019-09-17-SF@BOS,1,"[Mike Yastrzemski, Brandon Belt, Evan Longoria...","[Andrew Benintendi, Xander Bogaerts, Rafael De...",0.730,0.760,Logan Webb,Nathan Eovaldi,1.000,0.560,1.760,1.550,0.340,0.210,109,"63 degrees, Partly Cloudy.",7:12 PM.
5,2019-09-17-PHI@ATL,2,"[Jean Segura, J.T. Realmuto, Bryce Harper, Rhy...","[Ronald AcuÃ±a Jr., Ozzie Albies, Freddie Freem...",0.820,0.910,Vince Velasquez,Dallas Keuchel,0.450,0.310,1.380,1.290,0.360,0.300,100,"90 degrees, Partly Cloudy.",7:22 PM.
6,2019-09-17-CHW@MIN,0,"[Leury GarcÃ­a, Tim Anderson, JosÃ© Abreu, Eloy ...","[Mitch Garver, Jorge Polanco, Nelson Cruz, Edd...",0.900,0.950,Ross Detwiler,MartÃ­n PÃ©rez,0.600,NaN,1.540,1.720,0.340,0.280,98,"86 degrees, Clear.",6:41 PM.
7,2019-09-17-SD@MIL,0,"[Greg Garcia, Nick Martini, Manny Machado, Eri...","[Trent Grisham, Yasmani Grandal, Mike Moustaka...",0.680,0.870,Chris Paddack,Brandon Woodruff,0.280,0.350,1.270,2.670,0.350,0.280,100,"69 degrees, Clear.",6:40 PM.
8,2019-09-17-WAS@STL,0,"[Trea Turner, Adam Eaton, Anthony Rendon, Juan...","[Dexter Fowler, Kolten Wong, Paul Goldschmidt,...",1.000,0.830,Patrick Corbin,Miles Mikolas,0.570,0.430,1.210,1.190,0.260,0.360,95,"86 degrees, Clear.",6:45 PM.
9,2019-09-17-CIN@CHC,4,"[Josh VanMeter, Joey Votto, Eugenio SuÃ¡rez, Ar...","[Ben Zobrist, Nick Castellanos, Kris Bryant, K...",0.720,0.980,Sonny Gray,Yu Darvish,0.240,0.310,1.010,0.870,0.330,0.390,98,"71 degrees, Partly Cloudy.",7:06 PM.


# Get Features

## 1. Team YTD First Inning Run %

In [ ]:
team_pct = pd.read_html('https://www.teamrankings.com/mlb/stat/yes-run-first-inning-pct?date=2023-06-06')[0]

## 2. Pitcher YTD Runs in First Inning / IP

In [ ]:
payload = {"strPlayerId":"all","strSplitArr":[44],"strGroup":"season","strPosition":"P","strType":"1","strStartDate":"2023-03-01","strEndDate":"2023-04-01","strSplitTeams":False,"dctFilters":[],"strStatType":"player","strAutoPt":"true","arrPlayerId":[],"strSplitArrPitch":[],"arrWxTemperature":None,"arrWxPressure":None,"arrWxAirDensity":None,"arrWxElevation":None,"arrWxWindSpeed":None}
url = 'https://www.fangraphs.com/api/leaders/splits/splits-leaders'
r = requests.post(url, json=payload)

In [ ]:
pitcher_df = pd.json_normalize(r.json()['data'])
pitcher_df

,Season,playerName,playerId,TeamNameAbb,TBF,H,2B,3B,R,ER,...,BB,IBB,HBP,SO,AVG,OBP,SLG,ERA,wOBA,G
0,2023,Zack Greinke,1943,KCR,6.0,2.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.400000,0.500000,0.400000,0.0,0.410165,1.0
1,2023,Clayton Kershaw,2036,LAD,3.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.000000,0.000000,0.000000,0.0,0.000000,1.0
2,2023,Corey Kluber,2429,BOS,6.0,1.0,0.0,0.0,1.0,1.0,...,2.0,0.0,0.0,2.0,0.250000,0.500000,1.000000,9.0,0.565987,1.0
3,2023,Lance Lynn,2520,CHW,3.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,1.0
4,2023,Max Scherzer,3137,NYM,3.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.333333,0.333333,0.333333,0.0,0.294180,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,2023,Hunter Gaddis,25636,CLE,5.0,2.0,1.0,0.0,1.0,1.0,...,0.0,0.0,0.0,1.0,0.400000,0.400000,0.600000,9.0,0.425262,1.0
66,2023,Nick Lodolo,26378,CIN,7.0,3.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,2.0,0.500000,0.571429,0.500000,9.0,0.477647,1.0
67,2023,Alek Manoah,26410,TOR,6.0,2.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,2.0,0.400000,0.500000,0.400000,9.0,0.410165,1.0
68,2023,Spencer Strider,27498,ATL,3.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,2.0,0.000000,0.000000,0.000000,0.0,0.000000,1.0


In [ ]:
pitcher_df[pitcher_df['playerName'] == 'Zack Greinke']['R'].item()

0.0

## 3. Top 4 batters OPS last 30 days

In [34]:
ops_url = 'https://www.fangraphs.com/api/leaders/major-league/data?age=&pos=all&stats=bat&lg=all&qual=1&season='+str(date.year)+'&season1='+str(date.year)+'&startdate='+str(date_minus_thirty_days)+'&enddate='+str(yesterday)+'&month=1000&hand=&team=0&pageitems=2000000000&pagenum=1&ind=0&rost=0&players=&type=8&postseason=&sortdir=default&sortstat=WAR'
r = requests.get(ops_url)
ops_df = pd.json_normalize(r.json()['data'])[['PlayerName', 'OPS']]
ops_df['cleanedName'] = ops_df['PlayerName'].apply(unidecode)
ops_df[ops_df['PlayerName'] == 'Aaron Judge']['OPS'].item()

1.07397986

## 4. Ballpark Factor

In [ ]:
# hardcoded from https://baseballsavant.mlb.com/leaderboard/statcast-park-factors
park_factor_df = pd.DataFrame(
    {'Team': {0: 'COL', 1: 'CIN', 2: 'BOS', 3: 'LAA', 4: 'PHI', 5: 'KC', 6: 'CHW', 7: 'LAD', 8: 'BAL', 9: 'ARI', 10: 'PIT', 11: 'MIL', 12: 'SF', 13: 'ATL', 14: 'WAS', 15: 'CLE', 16: 'TOR', 17: 'MIA', 18: 'TEX', 19: 'NYY', 20: 'CHC', 21: 'HOU', 22: 'MIN', 23: 'DET', 24: 'TB', 25: 'NYM', 26: 'STL', 27: 'OAK', 28: 'SD', 29: 'SEA'},
     'Park Factor': {0: 112, 1: 111, 2: 109, 3: 104, 4: 104, 5: 103, 6: 102, 7: 102, 8: 101, 9: 101, 10: 100, 11: 100, 12: 100, 13: 100, 14: 100, 15: 99, 16: 99, 17: 99, 18: 99, 19: 99, 20: 98, 21: 98, 22: 98, 23: 97, 24: 96, 25: 96, 26: 95, 27: 94, 28: 94, 29: 91}}
)
park_factor_dict = park_factor_df.set_index('Team').to_dict()['Park Factor']

## 5. Pitcher WHIP Last 60 Days

In [36]:
whip_url = 'https://www.fangraphs.com/api/leaders/major-league/data?age=0&pos=all&stats=pit&lg=all&qual=1&season='+str(date.year)+'&season1='+str(date.year)+'&startdate='+str(date_minus_sixty_days)+'&enddate='+str(yesterday)+'&month=1000&hand=&team=0&pageitems=100000&pagenum=1&ind=0&rost=0&players=0&type=1&postseason=&sortdir=default&sortstat=SIERA&qual=1'
r = requests.get(whip_url)
whip_df = pd.json_normalize(r.json()['data'])
whip_df['cleanedName'] = whip_df['PlayerName'].apply(unidecode)
whip_df[whip_df['cleanedName'] == 'Dean Kremer']['WHIP'].item()

1.2452830636828103

## 7. Game Start Time

In [40]:
games = statsapi.schedule(start_date=date.strftime('%m/%d/%Y'))
game = {}
g = games[0]
game_id = g['game_id']
boxscore = statsapi.boxscore_data(game_id)
game_box_info = boxscore['gameBoxInfo']
for line in game_box_info:
    if(line['label'] == 'Weather'):
        game['weather'] = line['value']
    if(line['label'] == 'First pitch'):
        game['time'] = line['value']
print(game['time'])


1:43 PM.


## 8. Weather

In [39]:
print(game['weather'])

72 degrees, Sunny.


# MLB-StatsAPI

In [85]:
statsapi.boxscore_data(565997)['awayBatters'][1]['personId']

457705

In [78]:
id = statsapi.boxscore_data(565997)['awayPitchers'][1]['personId']
statsapi.get('people', {'personIds':id})['people'][0]['fullName']

'Vince Velasquez'

In [74]:
def get_lineup(game_id, home_away):
    hitters, i = 0, 1
    lineup = []
    box = statsapi.boxscore_data(game_id)[home_away+'Batters']
    while hitters < 4:
        id = box[i]['personId']
        sub = box[i]['substitution']
        if(not sub):
            hitters += 1
            name = statsapi.get('people', {'personIds':id})['people'][0]['fullName']
            lineup.append(name)
        i += 1
    return lineup

get_lineup(565997, 'away')

['Andrew McCutchen', 'J.T. Realmuto', 'Bryce Harper', 'Rhys Hoskins']

In [71]:
home_lineup

['Jeff McNeil',
 'Michael Conforto',
 'Robinson Cano',
 'Wilson Ramos',
 'Dominic Smith',
 'Todd Frazier']

In [49]:
statsapi.get('people', {'personIds':457705})['people'][0]['fullName']

'Andrew McCutchen'

In [84]:
print(statsapi.player_stats(457705))

Andrew "Drusneeze" McCutchen, DH (2009-)

Season Fielding (RF)
gamesPlayed: 8
gamesStarted: 7
assists: 0
putOuts: 19
errors: 0
chances: 19
fielding: 1.000
rangeFactorPerGame: 2.38
rangeFactorPer9Inn: 2.67
innings: 64.2
games: 8
doublePlays: 0
triplePlays: 0
throwingErrors: 0

Season Fielding (DH)
gamesPlayed: 98
gamesStarted: 97
assists: 0
putOuts: 0
errors: 0
chances: 0
fielding: .000
rangeFactorPerGame: 0.00
rangeFactorPer9Inn: -.--
innings: 0.0
games: 98
doublePlays: 0
triplePlays: 0
throwingErrors: 0

Season Hitting
gamesPlayed: 112
groundOuts: 91
airOuts: 103
runs: 55
doubles: 19
triples: 0
homeRuns: 12
strikeOuts: 100
baseOnBalls: 75
intentionalWalks: 1
hits: 100
hitByPitch: 4
avg: .256
atBats: 390
obp: .378
slg: .397
ops: .775
caughtStealing: 3
stolenBases: 11
stolenBasePercentage: .786
groundIntoDoublePlay: 8
numberOfPitches: 2008
plateAppearances: 473
totalBases: 155
rbi: 43
leftOnBase: 157
sacBunts: 0
sacFlies: 4
babip: .312
groundOutsToAirouts: 0.88
catchersInterference: 0
a

In [73]:
print(statsapi.linescore(565997))

Final    1 2 3 4 5 6 7 8 9  R   H   E  
Phillies 1 0 0 0 0 0 0 3 2  6   10  0  
Mets     0 0 0 0 0 0 0 0 0  0   6   3  


In [46]:
linescore = statsapi.linescore(565997)
top1 = int(re.search('[0-9]+', linescore.split('\n')[1]).group())
bot1 = int(re.search('[0-9]+', linescore.split('\n')[2]).group())

In [47]:
print(top1, bot1)

1 0


In [44]:
int(re.search('[0-9]+', statsapi.linescore(565997).split('\n')[1]).group())
int(re.search('[0-9]+', statsapi.linescore(565997).split('\n')[2]).group())

0

In [91]:
game_box_info = statsapi.boxscore_data(565997)['gameBoxInfo']
for line in game_box_info:
    if(line['label'] == 'Weather'):
        print(line['value'])
    if(line['label'] == 'First pitch'):
        print(line['value'])

66 degrees, Clear.
7:11 PM.


In [16]:
statsapi.get('sports', params={})
# MLB = sports 1
statsapi.get('seasons', params={'sportId':1})
# seasonId = year
statsapi.get('season', params={'seasonId':2022, 'sportId':1})
statsapi.get('gamePace', params={'season':2022})

{'copyright': 'Copyright 2023 MLB Advanced Media, L.P.  Use of any content on this page acknowledges agreement to the terms posted here http://gdx.mlb.com/components/copyright.txt',
 'teams': [],
 'leagues': [],
 'sports': [{'hitsPer9Inn': 16.58,
   'runsPer9Inn': 8.7,
   'pitchesPer9Inn': 296.08,
   'plateAppearancesPer9Inn': 76.07,
   'hitsPerGame': 16.33,
   'runsPerGame': 8.57,
   'inningsPlayedPerGame': 8.89,
   'pitchesPerGame': 291.58,
   'pitchersPerGame': 8.59,
   'plateAppearancesPerGame': 74.92,
   'totalGameTime': '7558:45:59',
   'totalInningsPlayed': 21604.0,
   'totalHits': 39674,
   'totalRuns': 20817,
   'totalPlateAppearances': 182052,
   'totalPitchers': 20883,
   'totalPitches': 708540,
   'totalGames': 2430,
   'total7InnGames': 10,
   'total9InnGames': 2204,
   'totalExtraInnGames': 216,
   'timePerGame': '03:06:38',
   'timePerPitch': '00:00:38',
   'timePerHit': '00:11:25',
   'timePerRun': '00:21:47',
   'timePerPlateAppearance': '00:02:29',
   'timePer9Inn': '

In [25]:
statsapi.schedule(start_date='05/01/2022')[0]['game_id']

663363

In [32]:
statsapi.get('game', params={'gamePk':663363})['gameData']['teams']['away']['abbreviation']

'BOS'

In [33]:
# get all abbreviations
for i in range(15):
    gameid = statsapi.schedule(start_date='05/01/2022')[i]['game_id']
    print(statsapi.get('game', params={'gamePk':gameid})['gameData']['teams']['away']['abbreviation'])
    print(statsapi.get('game', params={'gamePk':gameid})['gameData']['teams']['home']['abbreviation'])

BOS
BAL
MIN
TB
SD
PIT
HOU
TOR
SEA
MIA
LAA
CWS
NYY
KC
CHC
MIL
AZ
STL
ATL
TEX
CIN
COL
WSH
SF
CLE
OAK
DET
LAD
PHI
NYM
